In [ ]:
from transformers import TimesformerForVideoClassification

In [11]:
from transformers import AutoImageProcessor, TimesformerModel

In [3]:
import torch
import numpy as np
import av
from pathlib import Path 

In [4]:
vod_title = "M8 vs. EDG - VALORANT Masters Santiago - SWISS"
video_path = Path(f"../../data/vods/{vod_title}.mp4")

In [ ]:
def read_video_pyav(container,indices):
    '''
    Decode the video with PyAV decoder.
    Args:
        container (`av.container.input.InputContainer`): PyAV container.
        indices (`list[int]`): List of frame indices to decode.
    Returns:
        result (np.ndarray): np array of decoded frames of shape (num_frames, height, width, 3).
    '''
    frames = []
    container.seek(0)
    start_index = indices[0]
    end_index = indices[-1]
    for i,frame in enumerate(container.decode(video=0)):
        if i > end_index:
            break
        if i >=start_index and i in indices:
            frames.append(frame)
    return np.stack([x.to_ndarray(format="rgb24") for x in frames])

def sample_frame_indices(clip_len, frame_sample_rate, seg_len):
    '''
    Sample a given number of frame indices from the video.
    Args:
        clip_len (`int`): Total number of frames to sample.
        frame_sample_rate (`int`): Sample every n-th frame.
        seg_len (`int`): Maximum allowed index of sample's last frame.
    Returns:
        indices (`list[int]`): List of sampled frame indices
    '''
    converted_len = int(clip_len * frame_sample_rate)
    end_idx = np.random.randint(converted_len, seg_len)
    start_idx = end_idx - converted_len
    indices = np.linspace(start_idx, end_idx, num=clip_len)
    indices = np.clip(indices, start_idx, end_idx - 1).astype(np.int64)
    return indices
def sample_indices_from_start(start_sec,fps,num_frames=8,sample_fps=4,offset_sec=0.0):
    """
    start_secから一定間隔でフレームindexを取る。

    Args:
        start_sec: ラウンド開始秒
        fps: 動画fps
        num_frames: 取るフレーム数
        sample_fps: 1秒あたり何枚取るか
        offset_sec: start_secから何秒後を起点にするか

    Returns:
        indices: フレームindexのnp.ndarray
    """
    start_frame = int((start_sec+offset_sec)*fps)
    step = max(1,int(round(fps/sample_fps)))
    indices = start_frame + np.arrage(num_frames) * step
    return indices.astype(np.int64)
   

In [ ]:
container = av.open(video_path)

indeces = sample_indices_from_start(start_sec=10,fps=fps,num_frames=8,sample_fps=4,offset_sec=5.0)
video = read_video_pyav(container,indeces)

image_processor = AutoImageProcessor.from_pretrained("MCG-NJU/videomae-base")
model = TimesformerForVideoClassification.from_pretrained(
    "facebook/timesformer-base-finetuned-k400",
    num_labels=2,
    ignore_mismatched_sizes=True,
)